In [0]:
# Imports 
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("EventhubToBronzeDelta").getOrCreate()

In [0]:
# Paths

BASE_PATH = "abfss://lakehouse@staqilakehouseen.dfs.core.windows.net"

BRONZE_RAW_DELTA_PATH = f"{BASE_PATH}/bronze_delta/air_quality_raw/"
BRONZE_STRUCTURED_DELTA_PATH = f"{BASE_PATH}/bronze_delta/air_quality_structured/"
SILVER_DELTA_PATH = f"{BASE_PATH}/silver_delta/air_quality/"
GOLD_DELTA_BASE_PATH = f"{BASE_PATH}/gold_delta/"
CHECKPOINT_DELTA_BASE_PATH = f"{BASE_PATH}/checkpoints_delta/"

In [0]:
# Secrets

eh_namespace = dbutils.secrets.get("aqi-secrets", "eventhub-namespace")
eh_name = dbutils.secrets.get("aqi-secrets", "eventhub-name")
eh_policy_name = dbutils.secrets.get("aqi-secrets", "eventhub-listen-policy")
eh_policy_key = dbutils.secrets.get("aqi-secrets", "eventhub-listen-key")

print("Namespace loaded:", eh_namespace is not None)
print("Event Hub name loaded:", eh_name is not None)
print("Policy name loaded:", eh_policy_name is not None)
print("Policy key loaded:", eh_policy_key is not None)

Namespace loaded: True
Event Hub name loaded: True
Policy name loaded: True
Policy key loaded: True


In [0]:
# Bootstrap Servers

eh_connection_string = (
    f"Endpoint=sb://{eh_namespace}.servicebus.windows.net/;"
    f"SharedAccessKeyName={eh_policy_name};"
    f"SharedAccessKey={eh_policy_key}"
)

eh_bootstrap_servers = f"{eh_namespace}.servicebus.windows.net:9093"

eh_sasl = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" '
    f'password="{eh_connection_string}";'
)

print("Bootstrap servers:", eh_bootstrap_servers)
print("Event Hub:", eh_name)
print("SASL config created:", eh_sasl is not None)

Bootstrap servers: [REDACTED].servicebus.windows.net:9093
Event Hub: [REDACTED]
SASL config created: True


In [0]:
raw_stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", eh_bootstrap_servers)
    .option("subscribe", eh_name)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", eh_sasl)
    .option("startingOffsets", "earliest")
    .load()
)

json_stream_df = raw_stream_df.selectExpr("CAST(value AS STRING) AS json_string")

checkpoint_path = (
    "abfss://lakehouse@staqilakehouseen.dfs.core.windows.net/"
    "checkpoints/eventhub_memory_test_clean_v4/"
)

query = (
    json_stream_df.writeStream
    .format("memory")
    .queryName("eventhub_json_test_clean")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

spark.sql("""
SELECT *
FROM eventhub_json_test_clean
LIMIT 20
""").show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|json_string                                                                                                                                                                                                                                                                                                         |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"source": "test", "message": "hello event hubs"}                 

In [0]:
# Spark readStream

raw_stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", eh_bootstrap_servers)
    .option("subscribe", eh_name)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", eh_sasl)
    .option("startingOffsets", "earliest")
    .load()
)

json_stream_df = raw_stream_df.selectExpr("CAST(value AS STRING) AS json_string")

In [0]:
# writeStream Query

query = (
    json_stream_df.writeStream
    .format("delta")
    .option("path", BRONZE_RAW_DELTA_PATH)
    .option("checkpointLocation", f"{CHECKPOINT_DELTA_BASE_PATH}/eventhub_to_bronze_delta_v1/")
    .outputMode("append")
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
spark.read.format("delta").load(BRONZE_RAW_DELTA_PATH).show(20, truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|json_string                                                                                                                                                                                                                                                                                                           |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"source": "openmeteo", "latitude": 37.9838, "longitude": 23

In [0]:
# Parse json_string

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

aqi_schema = StructType(
    [
        StructField("source", StringType()),
        StructField("latitude", DoubleType()),
        StructField("longitude", DoubleType()),
        StructField("city_id", IntegerType()),
        StructField("city", StringType()),
        StructField("country", StringType()),
        StructField("measurement_timestamp", StringType()),
        StructField("european_aqi", IntegerType()),
        StructField("pm10", DoubleType()),
        StructField("pm2_5", DoubleType()),
        StructField("nitrogen_dioxide", DoubleType()),
        StructField("ingestion_timestamp_utc", StringType())
    ])

bronze_raw_df = spark.read.format("delta").load(BRONZE_RAW_DELTA_PATH)


parsed_df = bronze_raw_df.select(
    F.from_json(F.col("json_string"), aqi_schema).alias("data")
)

final_df = (
    parsed_df
    .select("data.*")
    .filter(F.col("source") == "openmeteo")
)

In [0]:
final_df.show()

+---------+--------+---------+-------+------------+-------+---------------------+------------+----+-----+----------------+-----------------------+
|   source|latitude|longitude|city_id|        city|country|measurement_timestamp|european_aqi|pm10|pm2_5|nitrogen_dioxide|ingestion_timestamp_utc|
+---------+--------+---------+-------+------------+-------+---------------------+------------+----+-----+----------------+-----------------------+
|openmeteo| 37.9838|  23.7275|      1|      Athens| Greece|     2026-07-09T00:00|          30|22.7| 15.3|            42.1|   2026-07-09T11:27:...|
|openmeteo| 37.9838|  23.7275|      1|      Athens| Greece|     2026-07-09T01:00|          30|23.5| 16.7|            44.3|   2026-07-09T11:27:...|
|openmeteo| 37.9838|  23.7275|      1|      Athens| Greece|     2026-07-09T04:00|          31|26.6| 16.6|            53.9|   2026-07-09T11:27:...|
|openmeteo| 37.9838|  23.7275|      1|      Athens| Greece|     2026-07-09T05:00|          31|27.4| 17.1|            5

In [0]:
# Write structured Bronze to Delta

final_df.write.format("delta").mode("overwrite").save(BRONZE_STRUCTURED_DELTA_PATH)

In [0]:
# Read Test

spark.read.format("delta").load(BRONZE_STRUCTURED_DELTA_PATH).show(20, truncate=False)

+---------+--------+---------+-------+------------+-------+---------------------+------------+----+-----+----------------+--------------------------------+
|source   |latitude|longitude|city_id|city        |country|measurement_timestamp|european_aqi|pm10|pm2_5|nitrogen_dioxide|ingestion_timestamp_utc         |
+---------+--------+---------+-------+------------+-------+---------------------+------------+----+-----+----------------+--------------------------------+
|openmeteo|37.9838 |23.7275  |1      |Athens      |Greece |2026-07-09T00:00     |30          |22.7|15.3 |42.1            |2026-07-09T11:27:23.032781+00:00|
|openmeteo|37.9838 |23.7275  |1      |Athens      |Greece |2026-07-09T01:00     |30          |23.5|16.7 |44.3            |2026-07-09T11:27:23.032781+00:00|
|openmeteo|37.9838 |23.7275  |1      |Athens      |Greece |2026-07-09T04:00     |31          |26.6|16.6 |53.9            |2026-07-09T11:27:23.032781+00:00|
|openmeteo|37.9838 |23.7275  |1      |Athens      |Greece |2026-